# ស៊ុមភាសាជំនួយ Microsoft — Azure OpenAI (Responses API)

ក្នុងឧទាហរណ៍កូដនេះ អ្នកនឹងប្រើប្រាស់ **Microsoft Agent Framework (MAF)** ដើម្បីបង្កើតភាសាជំនួយសាមញ្ញដែលគាំទ្រ​ដោយ **Azure OpenAI** ដោយប្រើ **Responses API**។

> **កំណត់ចំណាំអំពីការផ្លាស់ប្តូរ:** ឧទាហរណ៍នេះមុននេះបានប្រើ Semantic Kernel ជាមួយ GitHub Models។ វាត្រូវបានផ្លាស់ប្តូរទៅ Microsoft Agent Framework ហើយ GitHub Models (ដែលបានបោះបង់ និងនឹងបញ្ឈប់នៅខែកក្កដា ឆ្នាំ ២០២៦) ត្រូវបានជំនួសដោយ Azure OpenAI ដែលគាំទ្រ Responses API។ `OpenAIChatClient` ក្នុង MAF ទាក់ទងទៅនឹងចំណុចចប់ដំណើរការ `/openai/v1/` នៅ Azure OpenAI និងប្រើ Responses API ជាប្រព័ន្ធស្ដង់ដារ។

គោលបំណងនៃឧទាហរណ៍នេះគឺដើម្បីបង្ហាញជំហានដែលនឹងត្រូវដាក់អនុវត្តនៅក្នុងឧទាហរណ៍កម្មវិធីបន្ថែមផ្សេងទៀតនៅពេលក្រោយពេលដែលអនុវត្តតំបន់ប្រតិបត្តិភាសាជំនួយជាចម្រង់ផ្សេងៗ។


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## នាំចូលកញ្ចប់ Python ដែលត្រូវការ


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## ការបង្កើតឧបករណ៍មួយ

នៅក្នុង Microsoft Agent Framework, **ឧបករណ៍** គឺជាអនុគមន៍ Python ងាយៗមួយដែលត្រូវបានតុបតែងជាមួយ `@tool` ដែលភ្នាក់ងារអាចហៅបាន។ ខាងក្រោមនេះយើងកំណត់ឧបករណ៍មួយដែលបង្រួមទីកន្លែងថ្ងៃឈប់សម្រាកដោយចៃដន្យ ហើយជៀសវាងការកំណត់ជារឿយៗនៃទីកន្លែងដដែល។  


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini")

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## ការបង្កើតភ្នាក់ងារ

នៅទីនេះ យើងបង្កើតភ្នាក់ងារដែលមានឈ្មោះ `TravelAgent`។

នៅក្នុងឧទាហរណ៍នេះ យើងប្រើសេចក្តីណែនាំមូលដ្ឋានយ៉ាងសាមញ្ញ។ សូមអនុញ្ញាតអ្នកកែប្រែសេចក្តីណែនាំទាំងនេះដើម្បីសង្កេតមើលថាតើឥរិយាបថរបស់ភ្នាក់ងារប្រែប្រួលយ៉ាងដូចម្តេច។


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## ការរត់ភ្នាក់ងារ

ឥឡូវនេះ យើងអាចរត់ភ្នាក់ងារ។ យើងបង្កើត `AgentSession` ដើម្បីឲ្យភ្នាក់ងារចងចាំការសន្ទនាក្នុងអំឡុងពេលប្ដូរថ្មីៗ បន្ទាប់មកផ្ញើ `user_inputs` ពីរដង។ ដំបូងសួរអំពីការធ្វើដំណើរ; រួចហើយពេលទីពីរប្រាប់ថាអ្នកប្រើប្រាស់មិនចូលចិត្តការផ្តល់អនុសាសន៍នោះទេ ហើយសុំម្ដងទៀត — ភ្នាក់ងារប្រើប្រវត្តិសម័យនៃសម័យសម័យផងដែរជាមួយឧបករណ៍ `get_random_destination` ដើម្បីឆ្លើយតប។

អ្នកអាចកែសម្រួលសារเหล่านี้ ដើម្បីសង្កេតមើលរបៀបដែលភ្នាក់ងារឆ្លើយតបខុសគ្នា។ ការឆ្លើយតបត្រូវបាន **ផ្សាយបន្តផ្ទាល់** តាមTokenមួយៗ។


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
